# 06 – Overfitting and Underfitting

Everything you've learned — probability, distributions, sampling, hypothesis testing — comes together when you build a machine learning model.

The single most important thing to understand when building models is the **bias-variance tradeoff**: the fundamental tension between models that are too simple (can't learn the pattern) and models that are too complex (learn the noise as well as the pattern).

These two failure modes are called **underfitting** and **overfitting**.

Understanding them will make every model decision you make more principled — how much data you need, how to choose model complexity, how to interpret your evaluation results, and what to do when your model isn't working.

## 1) The Core Problem — Generalisation

A machine learning model's purpose is to **generalise** — to perform well on new, unseen data, not just the data it was trained on.

Memorising training data is not learning. A student who memorises past exam papers exactly will struggle when questions are phrased slightly differently. A model that memorises training data will struggle on new data.

We measure generalisation by:
- **Training error** — how well the model fits the data it learned from
- **Test error** — how well it performs on data it has never seen

The goal is low test error — not low training error.

```
Underfitting:  High training error + High test error
                → model is too simple to learn the pattern

Overfitting:   Low training error  + High test error
                → model memorised training data, can't generalise

Good fit:      Low training error  + Low test error
                → model learned the true pattern
```

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)

# Generate synthetic data: true pattern is a sine curve + noise
n = 50
X = np.sort(rng.uniform(0, 2 * np.pi, n))
y_true = np.sin(X)
y = y_true + rng.normal(0, 0.3, n)   # add noise

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x_plot = np.linspace(0, 2 * np.pi, 300)

# Degree 1: straight line — underfitting
coeffs1 = np.polyfit(X, y, deg=1)
y_pred1  = np.polyval(coeffs1, x_plot)
axes[0].scatter(X, y, color="steelblue", alpha=0.6, s=40, label="Data")
axes[0].plot(x_plot, np.sin(x_plot), color="black",     linestyle="--", linewidth=2, label="True pattern")
axes[0].plot(x_plot, y_pred1,         color="firebrick", linewidth=2.5, label="Degree 1 (underfit)")
train_err1 = np.mean((y - np.polyval(coeffs1, X))**2)
axes[0].set_title(f"Underfitting (Degree 1)Train MSE = {train_err1:.3f}", fontweight="bold")
axes[0].legend(fontsize=9); axes[0].grid(linestyle="--", alpha=0.3)

# Degree 3: just right
coeffs3 = np.polyfit(X, y, deg=3)
y_pred3  = np.polyval(coeffs3, x_plot)
axes[1].scatter(X, y, color="steelblue", alpha=0.6, s=40, label="Data")
axes[1].plot(x_plot, np.sin(x_plot), color="black",     linestyle="--", linewidth=2, label="True pattern")
axes[1].plot(x_plot, y_pred3,         color="seagreen",  linewidth=2.5, label="Degree 3 (good fit)")
train_err3 = np.mean((y - np.polyval(coeffs3, X))**2)
axes[1].set_title(f"Good Fit (Degree 3)Train MSE = {train_err3:.3f}", fontweight="bold")
axes[1].legend(fontsize=9); axes[1].grid(linestyle="--", alpha=0.3)

# Degree 15: overfit
coeffs15 = np.polyfit(X, y, deg=15)
y_pred15  = np.polyval(coeffs15, x_plot)
axes[2].scatter(X, y, color="steelblue", alpha=0.6, s=40, label="Data")
axes[2].plot(x_plot, np.sin(x_plot), color="black",     linestyle="--", linewidth=2, label="True pattern")
axes[2].plot(x_plot, y_pred15,        color="darkorange", linewidth=2.5, label="Degree 15 (overfit)")
train_err15 = np.mean((y - np.polyval(coeffs15, X))**2)
axes[2].set_title(f"Overfitting (Degree 15)Train MSE = {train_err15:.3f}", fontweight="bold")
axes[2].set_ylim(-3, 3)
axes[2].legend(fontsize=9); axes[2].grid(linestyle="--", alpha=0.3)

for ax in axes:
    ax.set_xlabel("X"); ax.set_ylabel("y")

fig.suptitle("Underfitting vs Good Fit vs Overfitting", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/06_stats/overfit_demo.png", dpi=120)
plt.close()
print("Overfitting demo plot saved.")
print()
print(f"Train MSE (degree 1)  = {train_err1:.4f}  ← underfitting")
print(f"Train MSE (degree 3)  = {train_err3:.4f}  ← good fit")
print(f"Train MSE (degree 15) = {train_err15:.4f}  ← overfitting (zero noise = memorised)")

**What you see in the charts:**

- **Degree 1 (underfit):** The straight line can't capture the wave shape at all. It's wrong everywhere. High training error because the model is too simple.

- **Degree 3 (good fit):** Follows the general wave shape without chasing every noisy point. Low training error and would generalise well to new data.

- **Degree 15 (overfit):** Passes through nearly every training point — train MSE approaches zero. But between training points, the curve goes wild. On new data, it would perform terribly.

## 2) Bias and Variance — The Mathematical Framing

The test error can be decomposed into three parts:

```
Total Error = Bias² + Variance + Irreducible Noise
```

**Bias:** Error from wrong assumptions in the model. A linear model trying to fit a sine curve has high bias — it fundamentally can't represent the true pattern.

**Variance:** Error from sensitivity to small fluctuations in training data. The degree-15 model's wild swings between training points are variance — if you trained it on a slightly different sample, you'd get a very different curve.

**Irreducible noise:** Random measurement error in the data. You can never eliminate this, no matter how good your model is.

```
Simple model  →  High bias,   Low variance   (underfit)
Complex model →  Low bias,    High variance  (overfit)
```

In [ ]:
# Demonstrate bias-variance tradeoff by fitting many different samples
n_experiments = 100
x_test = np.linspace(0, 2 * np.pi, 100)
y_test_true = np.sin(x_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, degree, title in zip(axes, [1, 3, 15],
                              ["Degree 1 (High Bias)", "Degree 3 (Balanced)", "Degree 15 (High Variance)"]):
    predictions = []
    for _ in range(n_experiments):
        X_exp = np.sort(rng.uniform(0, 2 * np.pi, 30))
        y_exp = np.sin(X_exp) + rng.normal(0, 0.3, 30)
        coeffs = np.polyfit(X_exp, y_exp, deg=degree)
        pred = np.polyval(coeffs, x_test)
        # clip for degree 15 to avoid plotting artefacts
        pred = np.clip(pred, -3, 3)
        predictions.append(pred)
        ax.plot(x_test, pred, alpha=0.08, color="steelblue", linewidth=1)
    
    mean_pred = np.mean(predictions, axis=0)
    ax.plot(x_test, mean_pred,    color="firebrick", linewidth=2.5, label="Mean prediction")
    ax.plot(x_test, y_test_true,  color="black",     linewidth=2, linestyle="--", label="True function")
    
    bias2   = np.mean((mean_pred - y_test_true)**2)
    var_val = np.mean(np.var(predictions, axis=0))
    ax.set_title(f"{title}\nBias²={bias2:.3f}  Var={var_val:.3f}", fontweight="bold")
    ax.set_xlabel("X"); ax.set_ylabel("y")
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=9); ax.grid(linestyle="--", alpha=0.3)

fig.suptitle("Bias-Variance Tradeoff (100 different samples each)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/06_stats/bias_variance.png", dpi=120)
plt.close()
print("Bias-variance plot saved.")

Each thin blue line is the model fit on one different sample. The red line is the average across all 100 fits.

- **Degree 1:** All lines cluster tightly (low variance) but are all wrong in the same way (high bias)
- **Degree 3:** Lines cluster close to the true curve — low bias AND low variance
- **Degree 15:** Lines scatter wildly (high variance) but on average they're closer to the truth (lower bias)

The **sweet spot** (Degree 3) has low bias AND low variance. This is what you're searching for when choosing model complexity.

## 3) The Learning Curve — Diagnosing Your Model

A **learning curve** plots training error and validation error as a function of training set size. It's the most direct tool for diagnosing over/underfitting.

In [ ]:
# Simulate learning curves for an underfit and overfit model

def get_learning_curve(degree, n_trials=30):
    sample_sizes = [10, 20, 30, 50, 75, 100, 150, 200]
    train_errors, val_errors = [], []

    X_all = np.sort(rng.uniform(0, 2 * np.pi, 250))
    y_all = np.sin(X_all) + rng.normal(0, 0.3, 250)

    for size in sample_sizes:
        tr_err, vl_err = [], []
        for _ in range(n_trials):
            idx = rng.choice(len(X_all), size=size, replace=False)
            X_tr, y_tr = X_all[idx], y_all[idx]
            mask_val   = np.ones(len(X_all), dtype=bool); mask_val[idx] = False
            X_val, y_val = X_all[mask_val][:50], y_all[mask_val][:50]

            coeffs = np.polyfit(X_tr, y_tr, deg=degree)
            tr_err.append(np.mean((y_tr  - np.polyval(coeffs, X_tr))**2))
            vl_err.append(np.mean((y_val - np.polyval(coeffs, X_val))**2))

        train_errors.append(np.mean(tr_err))
        val_errors.append(np.mean(vl_err))

    return sample_sizes, train_errors, val_errors

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
configs = [(1, "Underfitting (Degree 1)"), (4, "Good Fit (Degree 4)"), (15, "Overfitting (Degree 15)")]

for ax, (deg, title) in zip(axes, configs):
    sizes, tr, vl = get_learning_curve(deg)
    ax.plot(sizes, tr, "o-", color="steelblue",  linewidth=2, markersize=5, label="Train error")
    ax.plot(sizes, vl, "s-", color="firebrick",  linewidth=2, markersize=5, label="Val error")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("MSE")
    ax.legend(); ax.grid(linestyle="--", alpha=0.4)
    ax.set_ylim(bottom=0)

fig.suptitle("Learning Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/06_stats/learning_curves.png", dpi=120)
plt.close()
print("Learning curves saved.")

**How to read learning curves:**

**Underfitting:**
- Both training and validation error are high
- The two lines are close together but both plateau at a high value
- **Fix:** Use a more complex model, add features, engineer better features

**Good fit:**
- Training error is low and validation error is close to it
- The gap between them is small
- Both curves converge as sample size increases

**Overfitting:**
- Training error is very low (nearly zero)
- Validation error is much higher — a large gap
- Gap may shrink with more data
- **Fix:** Get more data, regularise, reduce model complexity, use dropout

## 4) Cross-Validation — A More Reliable Estimate of Test Error

A single train/test split is noisy — your score depends on which examples happened to end up in the test set.

**K-fold cross-validation** divides the data into K equal parts (folds), trains on K-1 folds, and tests on the remaining fold. This is repeated K times, with each fold serving as the test set once.

```
Fold 1:  [TEST ] [train] [train] [train] [train]
Fold 2:  [train] [TEST ] [train] [train] [train]
Fold 3:  [train] [train] [TEST ] [train] [train]
Fold 4:  [train] [train] [train] [TEST ] [train]
Fold 5:  [train] [train] [train] [train] [TEST ]
```

Final score = average of all 5 test scores. Much more stable than a single split.

In [ ]:
# Manual k-fold cross-validation
def kfold_mse(X, y, degree, k=5):
    n = len(X)
    fold_size = n // k
    indices = rng.permutation(n)   # shuffle once
    mses = []
    for fold in range(k):
        val_idx   = indices[fold * fold_size : (fold + 1) * fold_size]
        train_idx = np.concatenate([indices[:fold * fold_size],
                                    indices[(fold + 1) * fold_size:]])
        X_tr, y_tr   = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx],   y[val_idx]
        coeffs = np.polyfit(X_tr, y_tr, deg=degree)
        mse    = np.mean((y_val - np.polyval(coeffs, X_val))**2)
        mses.append(mse)
    return np.mean(mses), np.std(mses)

X_data = np.sort(rng.uniform(0, 2 * np.pi, 150))
y_data = np.sin(X_data) + rng.normal(0, 0.3, 150)

print("5-Fold Cross-Validation MSE by Polynomial Degree:")
print("-" * 50)
best_deg, best_mse = 1, float("inf")
for deg in [1, 2, 3, 4, 5, 8, 12, 15]:
    mean_mse, std_mse = kfold_mse(X_data, y_data, deg)
    marker = " ← BEST" if mean_mse < best_mse else ""
    if mean_mse < best_mse:
        best_mse, best_deg = mean_mse, deg
        marker = " ← BEST"
    else:
        marker = ""
    print(f"  Degree {deg:>2}: CV MSE = {mean_mse:.4f} ± {std_mse:.4f}{marker}")

print()
print(f"Best degree: {best_deg} (lowest cross-validation error)")

Cross-validation picks the degree (model complexity) with the lowest *average test error across folds* — not the lowest training error. This is how you choose hyperparameters in any model.

**In scikit-learn** (which you'll use in the next section), this is done with `cross_val_score` and `GridSearchCV` — but the underlying logic is exactly this.

## 5) Regularisation — Making Complex Models Behave

Instead of reducing complexity (fewer features, lower degree), you can **penalise** the model for being too complex. This is called regularisation.

For linear models, two common approaches:

- **Ridge (L2):** Penalise the sum of squared coefficients. Shrinks all coefficients toward zero.
- **Lasso (L1):** Penalise the sum of absolute coefficients. Can shrink some to exactly zero — effectively selecting features.

**The regularisation parameter λ (or C in scikit-learn):**
- Large λ → strong penalty → simpler model → more bias, less variance
- Small λ → weak penalty → complex model → less bias, more variance

In [ ]:
# Demonstrate regularisation conceptually with polynomial features
# Add L2 regularisation manually to polynomial fitting

from numpy.linalg import lstsq

def poly_features(X, degree):
    return np.column_stack([X**i for i in range(degree + 1)])

def ridge_fit(X_tr, y_tr, degree, lam):
    Phi = poly_features(X_tr, degree)
    n_feat = Phi.shape[1]
    # Ridge: (PhiT Phi + lam I) w = PhiT y
    A = Phi.T @ Phi + lam * np.eye(n_feat)
    b = Phi.T @ y_tr
    w = np.linalg.solve(A, b)
    return w

X_train = np.sort(rng.uniform(0, 2 * np.pi, 30))
y_train = np.sin(X_train) + rng.normal(0, 0.3, 30)
x_plot  = np.linspace(0, 2 * np.pi, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lambdas = [0.0001, 1.0, 100.0]
labels  = ["λ=0.0001 (weak penalty)", "λ=1 (balanced)", "λ=100 (strong penalty)"]

for ax, lam, label in zip(axes, lambdas, labels):
    w = ridge_fit(X_train, y_train, degree=15, lam=lam)
    y_plot = poly_features(x_plot, 15) @ w
    y_train_pred = poly_features(X_train, 15) @ w

    ax.scatter(X_train, y_train, color="steelblue", s=40, alpha=0.6, label="Training data")
    ax.plot(x_plot, np.sin(x_plot), color="black",    linestyle="--", linewidth=2, label="True function")
    ax.plot(x_plot, np.clip(y_plot, -3, 3), color="firebrick", linewidth=2.5, label=label)

    train_mse = np.mean((y_train - y_train_pred)**2)
    ax.set_title(f"Ridge Regularisation\n{label}\nTrain MSE={train_mse:.3f}", fontweight="bold")
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=8); ax.grid(linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("/home/claude/06_stats/regularisation.png", dpi=120)
plt.close()
print("Regularisation plot saved.")

**What you see:**

- **Weak penalty (λ=0.0001):** Still overfitting — the penalty is too small to constrain the degree-15 polynomial
- **Balanced (λ=1):** The wild oscillations are tamed — model is complex but regularised
- **Strong penalty (λ=100):** Forced toward a very simple shape — approaching underfitting

Finding the right λ is exactly the same problem as finding the right model complexity — and the solution is the same: cross-validation.

## 6) Practical Checklist

When a model isn't working, diagnose it:

**High training AND validation error → Underfitting**
- Try a more complex model
- Add more or better features (feature engineering)
- Reduce regularisation strength

**Low training error, high validation error → Overfitting**
- Get more training data (most effective fix if feasible)
- Add regularisation (Ridge, Lasso, dropout, etc.)
- Use a simpler model
- Remove noisy or redundant features
- Use cross-validation to select hyperparameters

**High variance in CV scores → small dataset or noisy data**
- Collect more data
- Use k-fold CV with higher k (e.g., k=10 or leave-one-out)

In [ ]:
# Summary: how errors behave with complexity
degrees = list(range(1, 16))
train_mselist, val_mselist = [], []

X_full = np.sort(rng.uniform(0, 2 * np.pi, 200))
y_full = np.sin(X_full) + rng.normal(0, 0.3, 200)

# 70/30 single split
split = int(0.7 * len(X_full))
X_tr, y_tr   = X_full[:split], y_full[:split]
X_val, y_val = X_full[split:], y_full[split:]

for deg in degrees:
    c = np.polyfit(X_tr, y_tr, deg=deg)
    train_mselist.append(np.mean((y_tr  - np.polyval(c, X_tr))**2))
    val_mselist.append(  np.mean((y_val - np.polyval(c, X_val))**2))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees, train_mselist, "o-", color="steelblue",  linewidth=2, markersize=6, label="Train MSE")
ax.plot(degrees, val_mselist,   "s-", color="firebrick",  linewidth=2, markersize=6, label="Validation MSE")
ax.axvline(degrees[np.argmin(val_mselist)], color="seagreen", linestyle="--", linewidth=2,
           label=f"Optimal degree = {degrees[np.argmin(val_mselist)]}")
ax.set_title("Model Complexity vs Error — The Classic U-Shape", fontweight="bold")
ax.set_xlabel("Polynomial Degree (Model Complexity →)")
ax.set_ylabel("Mean Squared Error")
ax.set_ylim(bottom=0, top=min(2, max(val_mselist)*1.2))
ax.legend()
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/06_stats/complexity_curve.png", dpi=120)
plt.close()
print("Complexity curve saved.")
print()
optimal = degrees[np.argmin(val_mselist)]
print(f"Optimal degree (min validation MSE): {optimal}")
print(f"Train MSE at optimal: {train_mselist[optimal-1]:.4f}")
print(f"Val   MSE at optimal: {val_mselist[optimal-1]:.4f}")

**The U-shaped validation curve** is the canonical picture of the bias-variance tradeoff:

- Left side (simple models): both training and validation error are high → underfitting zone
- The minimum of the validation curve: the sweet spot
- Right side (complex models): training error continues to fall but validation error rises → overfitting zone

This shape appears in essentially every hyperparameter tuning problem in machine learning.

## Summary

| Situation | Bias | Variance | Training Err | Validation Err | Fix |
|---|---|---|---|---|---|
| Underfitting | High | Low | High | High | More complexity, better features |
| Good fit | Low | Low | Low | Low | Done ✓ |
| Overfitting | Low | High | Low | High | More data, regularisation, simplify |

**Key tools:**
- **Learning curves** — diagnose the problem
- **Cross-validation** — choose model complexity honestly
- **Regularisation (Ridge/Lasso)** — constrain complex models
- **Effect size thinking** — practical significance matters more than a single metric

**The connection to statistics:**
- Bias = systematic error (like a biased estimator)
- Variance = sampling variability (like the standard error)
- Cross-validation error is your empirical estimate of expected test error

Next section: **Machine Learning** — where all of this becomes real.

## Practice Questions 1

1. Explain in your own words why a model with zero training error might still be a bad model.

2. Sketch (describe in words) what a learning curve looks like for:
   a) A model that is underfitting
   b) A model that is overfitting
   c) A model that just needs more data

3. Using the data from this notebook, fit polynomial degrees 1 through 10. Plot both training and validation MSE vs degree. Identify the degree with the lowest validation MSE.

## Practice Questions 2

1. You're building a spam classifier. It has 99% training accuracy and 72% test accuracy on the same dataset.
   - Is this overfitting, underfitting, or good fit?
   - What three things would you try to fix it?

2. You have a polynomial of degree 10 that is overfitting. You add Ridge regularisation with λ = 10. What do you expect to happen to:
   - Training error?
   - Validation error?
   - The number of non-zero coefficients?

3. A model trained on 500 examples has validation MSE = 2.4 ± 0.8 (mean ± std across 5 folds). A simpler model achieves 2.6 ± 0.3. Which model would you prefer in production, and why?